In [5]:
import torch
import torch.nn as nn
input_size = 10
output_size = 6

class FunctionEmulator(nn.Module):
    def __init__(self):
        super(FunctionEmulator, self).__init__()
        # Input (input_size) -> Hidden (128) -> Hidden (64) -> Output (output_size)
        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, output_size) 
        )

    def forward(self, x):
        return self.network(x)

model = FunctionEmulator()

In [6]:
# Load CSV, prepare DataLoaders, train model (expects `model`, input_size, output_size defined)
import pandas as pd
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
import joblib


In [7]:

FILE = '/home/Niki.Zadeh/platforms_CEFI/mom6/MOM6SIS2_experiments/MOM6SIS2COBALT.global_twodegree/co2_training_dump.csv'   # adjust path if needed

# Read CSV
df = pd.read_csv(FILE)
ncols = df.shape[1]
expected = input_size + output_size
if ncols != expected:
    raise ValueError(f'CSV has {ncols} cols but expected input_size+output_size = {expected}')

df.head()  # check data structure

,Temp,Sal,dic,po4,sio4,alk,htotallo,htotalhi,htotal,zt,co2star,alpha,pCO2surf,co3_ion,omega_arag,omega_calc
0,-0.4065,34.16,0.002055,0.000002,0.000062,0.002317,4.275000e-11,4.275000e-07,4.284000e-09,2.005,0.000011,0.06384,168.2,0.000176,2.659,4.239
1,-0.2555,34.12,0.002058,0.000002,0.000062,0.002315,4.396000e-11,4.396000e-07,4.402000e-09,2.001,0.000011,0.06348,173.7,0.000173,2.610,4.160
2,-0.4104,34.12,0.002063,0.000002,0.000062,0.002312,4.508000e-11,4.508000e-07,4.515000e-09,2.002,0.000011,0.06387,178.7,0.000168,2.541,4.051
3,-0.4771,34.12,0.002066,0.000002,0.000063,0.002310,4.611000e-11,4.611000e-07,4.620000e-09,1.989,0.000012,0.06404,183.4,0.000165,2.486,3.963
4,0.5447,33.87,0.002067,0.000001,0.000061,0.002345,4.127000e-11,4.127000e-07,4.141000e-09,1.999,0.000010,0.06163,164.2,0.000188,2.837,4.522


In [12]:
# Compute per-column mean and spread RMS (population std)
import numpy as np
arr = df.values.astype(np.float64)
colnames = list(df.columns)
means = np.mean(arr, axis=0)
spread_rms = np.sqrt(np.mean((arr - means)**2, axis=0))
min = np.min(arr, axis=0)
max = np.max(arr, axis=0)
print('col, mean, spread_rms, min, max')
for name, m, s, r, R in zip(colnames, means, spread_rms, min, max):
    print(f'{name}: mean={m:.6e}, spread_rms={s:.6e}, min={r:.6e}, max={R:.6e}')

col, mean, spread_rms, min, max
Temp: mean=4.975764e+00, spread_rms=7.564329e+00, min=-2.164000e+00, max=3.065000e+01
Sal: mean=3.432358e+01, spread_rms=1.633475e+00, min=5.593000e+00, max=3.742000e+01
dic: mean=2.147478e-03, spread_rms=1.233640e-04, min=1.122000e-03, max=2.401000e-03
po4: mean=1.549955e-06, spread_rms=7.980217e-07, min=4.421000e-09, max=3.852000e-06
sio4: mean=5.246743e-05, spread_rms=5.125618e-05, min=2.961000e-07, max=2.778000e-04
alk: mean=2.317126e-03, spread_rms=7.090888e-05, min=1.167000e-03, max=2.603000e-03
htotallo: mean=3.878167e+08, spread_rms=4.297844e+11, min=1.094000e-11, max=7.388000e+14
htotalhi: mean=3.878167e+12, spread_rms=4.297844e+15, min=1.094000e-07, max=7.388000e+18
htotal: mean=1.057590e-08, spread_rms=4.725639e-09, min=1.093000e-09, max=6.527000e-08
zt: mean=1.142651e+03, spread_rms=1.509317e+03, min=1.610000e+00, max=6.501000e+03
co2star: mean=-9.865680e+03, spread_rms=1.146862e+03, min=-9.999000e+03, max=1.548000e-04
alpha: mean=-9.865679e+

In [8]:

data = df.values.astype(np.float32)
X = data[:, :input_size]
y = data[:, input_size: input_size+output_size]

# Training hyperparameters
epochs = 30
batch_size = 1024
lr = 1e-3
val_frac = 0.1
test_frac = 0.1  # fraction to hold out as test set
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Subsample large dataset for quicker development runs
max_samples = 100000  # set to None or 0 to use entire dataset
N_total = X.shape[0]
if (max_samples is not None) and (max_samples > 0) and (N_total > max_samples):
    sel = np.random.choice(N_total, size=max_samples, replace=False)
    X = X[sel]
    y = y[sel]

# Shuffle and split (test set reserved first)
N = X.shape[0]
idx = np.random.permutation(N)
n_test = max(1, int(test_frac * N)) if (test_frac is not None and test_frac>0) else 0
n_val = max(1, int(val_frac * N))
test_idx = idx[:n_test]
val_idx = idx[n_test:n_test+n_val]
train_idx = idx[n_test+n_val:]

# Fit scalers on training data and transform
x_scaler = StandardScaler()
y_scaler = StandardScaler()
x_scaler.fit(X[train_idx])
y_scaler.fit(y[train_idx])
# Save scalers for later inference
joblib.dump(x_scaler, 'x_scaler.pkl')
joblib.dump(y_scaler, 'y_scaler.pkl')

X_scaled = x_scaler.transform(X)
y_scaled = y_scaler.transform(y)

# Prepare tensors (move to device later)
X_train = torch.from_numpy(X_scaled[train_idx]).float()
y_train = torch.from_numpy(y_scaled[train_idx]).float()
X_val = torch.from_numpy(X_scaled[val_idx]).float()
y_val = torch.from_numpy(y_scaled[val_idx]).float()

X_train.shape, y_train.shape, X_val.shape, y_val.shape  # sanity check

(torch.Size([80000, 10]),
 torch.Size([80000, 6]),
 torch.Size([10000, 10]),
 torch.Size([10000, 6]))

In [9]:
X_train[0,:]

tensor([ 2.1773,  0.5618, -1.1070, -0.9303, -0.9450,  0.4622, -0.7249, -0.7249,
        -0.7277, -0.7413])

In [10]:

# Move to device inside DataLoader loop or to tensors here
X_train = X_train.to(device)
y_train = y_train.to(device)
X_val = X_val.to(device)
y_val = y_val.to(device)

# Prepare test set (if any) and DataLoaders
if 'test_idx' in globals() and (test_idx is not None) and (len(test_idx) > 0):
    X_test = torch.from_numpy(X_scaled[test_idx]).float().to(device)
    y_test = torch.from_numpy(y_scaled[test_idx]).float().to(device)
    test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, shuffle=False)
else:
    test_loader = None

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=batch_size, shuffle=False)

# Setup model, loss, optimizer
model = model.to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

# Training loop
for ep in range(1, epochs+1):
    model.train()
    running = 0.0
    count = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        running += loss.item() * xb.size(0)
        count += xb.size(0)
    train_loss = running / count

    model.eval()
    with torch.no_grad():
        vrunning = 0.0
        vcount = 0
        for xb, yb in val_loader:
            preds = model(xb)
            vloss = criterion(preds, yb)
            vrunning += vloss.item() * xb.size(0)
            vcount += xb.size(0)
        val_loss = vrunning / vcount

    print(f'Epoch {ep:3d}  Train MSE: {train_loss:.6e}  Val MSE: {val_loss:.6e}')

# Save model
torch.save({'model_state_dict': model.state_dict(),
            'input_size': input_size,
            'output_size': output_size}, 'model.pth')
print('Saved model.pth')

# Evaluate on test set (if reserved): compute MSE, RMSE, and fraction within 5% relative error
if test_loader is not None:
    model.eval()
    preds_list = []
    y_list = []
    with torch.no_grad():
        for xb, yb in test_loader:
            p = model(xb)
            preds_list.append(p.cpu().numpy())
            y_list.append(yb.cpu().numpy())
    preds_arr = np.vstack(preds_list)
    y_arr = np.vstack(y_list)
    # inverse transform to physical units
    preds_phys = y_scaler.inverse_transform(preds_arr)
    y_phys = y_scaler.inverse_transform(y_arr)
    mse = np.mean((preds_phys - y_phys)**2)
    rmse = np.sqrt(mse)
    denom = np.maximum(np.abs(y_phys), 1e-12)
    rel_err = np.abs((preds_phys - y_phys) / denom)
    acc_5pct = np.mean(rel_err < 0.05)
    print(f'Test MSE: {mse:.6e}  RMSE: {rmse:.6e}  Acc(<5% rel): {acc_5pct:.4f}')


Epoch   1  Train MSE: 5.710758e-01  Val MSE: 5.269585e-01
Epoch   2  Train MSE: 4.897111e-01  Val MSE: 5.251118e-01
Epoch   3  Train MSE: 4.872276e-01  Val MSE: 5.219376e-01
Epoch   4  Train MSE: 4.862033e-01  Val MSE: 5.222771e-01
Epoch   5  Train MSE: 4.853022e-01  Val MSE: 5.216478e-01
Epoch   6  Train MSE: 4.854445e-01  Val MSE: 5.208165e-01
Epoch   7  Train MSE: 4.852123e-01  Val MSE: 5.210053e-01
Epoch   8  Train MSE: 4.846111e-01  Val MSE: 5.201867e-01
Epoch   9  Train MSE: 4.837193e-01  Val MSE: 5.207476e-01
Epoch  10  Train MSE: 4.835769e-01  Val MSE: 5.197875e-01
Epoch  11  Train MSE: 4.833215e-01  Val MSE: 5.207347e-01
Epoch  12  Train MSE: 4.833777e-01  Val MSE: 5.209871e-01
Epoch  13  Train MSE: 4.826722e-01  Val MSE: 5.188600e-01
Epoch  14  Train MSE: 4.824026e-01  Val MSE: 5.197420e-01
Epoch  15  Train MSE: 4.821897e-01  Val MSE: 5.189613e-01
Epoch  16  Train MSE: 4.816332e-01  Val MSE: 5.203284e-01
Epoch  17  Train MSE: 4.815916e-01  Val MSE: 5.182898e-01
Epoch  18  Tra